# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 5: Prompt Engineering Challenge - First TextWorld Problems</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know what TextWorld is and how an LLM plays a text game step by step, <br>
        ... can evaluate a prompt on training, validation, and test games, <br>
        ... can apply what you learned in 4.01 to 4.04 to a task with many steps.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [Prompt Engineering Guide](https://www.promptingguide.ai/)
* [Côté et al. (2018): TextWorld: A Learning Environment for Text-based Games](https://doi.org/10.48550/arXiv.1806.11532)
* [Microsoft Research Blog: First TextWorld Problems](https://www.microsoft.com/en-us/research/blog/first-textworld-problems-the-competition-using-text-based-games-to-advance-capabilities-of-ai-agents/)

Documentation for this notebook:

* [TextWorld Documentation](https://textworld.readthedocs.io/en/stable/notes/framework.html) and [TextWorld on GitHub](https://github.com/microsoft/TextWorld)
* [LangChain Docs: ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)

## Where We Are

* **4.01 to 4.04:** prompts for one question and one answer.
* **Now:** a task with **many steps**. The model plays a text game: it reads the state of the game, gives one command, reads the new state, and so on.

Every step is one request with the techniques you know: a system prompt, a structured prompt, and the conversation so far.

## Setup

The same setup as in 4.03: run the next cell and choose where the model runs.

| `provider` | Where the model runs | `model_name` | Good to know |
|---|---|---|---|
| `"ollama"` | **Ollama** on your computer | `qwen3.5:9b`, `qwen3.5:4b`, `qwen3.5:2b` | free, no account; install [Ollama](https://ollama.com/download) first, then download the model once in a terminal: `ollama pull qwen3.5:9b` (6.6 GB; 4b: 3.3 GB; 2b: 2.7 GB); without a GPU, it runs on the CPU, which is slower |
| `"colab"` | **Ollama on Google Colab** | the same three | free, needs a Google account; the cell installs Ollama in the Colab runtime and downloads the model, as in 4.01; choose a GPU runtime first (Runtime > Change runtime type > T4 GPU) |
| `"huggingface"` | **Hugging Face endpoints** of the course | the same three, one endpoint each | key and addresses from the course; if an endpoint was asleep, the first request takes 1 to 2 minutes |
| `"ai-gateway"` | **AI-Gateway** of Paderborn University | `gwdg.qwen3.8-27b-no-think` (thinking off), `gwdg.qwen3.8-27b` (thinking on) | your university key (see [setup/SETUP_AI_GATEWAY.md](https://github.com/olivermueller/amlta-2026/blob/main/setup/SETUP_AI_GATEWAY.md)); only in the university network or the VPN, not on Colab; about 10 requests per minute |
| `"other"` | **another** OpenAI-compatible API | the name your provider uses | base URL and key in `.env` (`OTHER_BASE`, `OTHER_KEY`) |

Keys and addresses come from the `.env` file in the repo folder (see `.env.example`). Without a `.env` file, for example on Colab, the cell asks for them.

**For this notebook:**

* **Model:** use `"model_name": "qwen3.5:9b"` with `ollama`, `colab`, or `huggingface`. The basic attempt below was made with it.
* **Number of requests:** one game takes up to 50 requests, one evaluation of all 20 training games up to 1,000. With the AI-Gateway (about 10 requests per minute), that takes hours. Use `ollama`, `colab`, or `huggingface`.
* **Windows:** TextWorld runs on Linux and macOS, not on Windows. On Windows, use WSL or Colab (provider `colab`).

In [ ]:
%pip install -q textworld termcolor langchain-openai python-dotenv tqdm

In [ ]:
# Run this cell: it only defines setup_model(). We call it before every ChatOpenAI, with the provider and the model.
import getpass
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()  # reads the .env file if there is one: this folder first, then the folders above (the repo folder)
IN_COLAB = "google.colab" in sys.modules

PROVIDERS = ["ollama", "colab", "huggingface", "ai-gateway", "other"]
QWEN = {  # model_name: (download size in Ollama, the name of its endpoint address in .env, the model on Hugging Face)
    "qwen3.5:9b": ("6.6 GB", "HF_GATEWAY_BASE_9B", "unsloth/Qwen3.5-9B-GGUF"),
    "qwen3.5:4b": ("3.3 GB", "HF_GATEWAY_BASE_4B", "unsloth/Qwen3.5-4B-GGUF"),
    "qwen3.5:2b": ("2.7 GB", "HF_GATEWAY_BASE_2B", "unsloth/Qwen3.5-2B-GGUF"),
}
GATEWAY_QWEN = {  # on the AI-Gateway, the model name decides about thinking
    "gwdg.qwen3.8-27b-no-think": "thinking off",
    "gwdg.qwen3.8-27b": "thinking on",
}


def value(env_name, question, secret=False):
    """Takes the value from .env, or asks for it (keys are typed hidden)."""
    found = os.getenv(env_name)
    if found:
        return found.strip()
    answer = getpass.getpass(question).strip() if secret else input(question).strip()
    os.environ[env_name] = answer  # asked once: the next call of setup_model() takes it from here
    return answer


def start_ollama(model, size, colab=False):
    """Makes sure that the Ollama server runs and has the model. Downloads the model only on Colab."""
    url = (os.getenv("OLLAMA_BASE") or "http://localhost:11434").rstrip("/")

    def running():
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except OSError:
            return False

    if colab and shutil.which("ollama") is None:
        # the same steps as in 4.01: the installer needs pciutils and zstd
        print("Installing Ollama in the Colab runtime (about a minute) ...")
        subprocess.run("sudo apt-get install -y -qq pciutils zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null",
                       shell=True, check=True)
    if not running():
        if shutil.which("ollama") is None:
            raise RuntimeError("Ollama is not installed. Install it from https://ollama.com/download and run this cell again.")
        # on Colab, the installer may say that the server is running, but it is not; so we start it ourselves (as in 4.01)
        print("Starting the Ollama server ...")
        ollama_log = open("ollama.log", "w")  # a pipe that nobody reads could fill up and block the server
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
        for _ in range(30):
            if running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("The Ollama server did not start. Look into ollama.log.")

    installed = [m["name"] for m in json.load(urllib.request.urlopen(url + "/api/tags"))["models"]]
    if model not in installed and not colab:  # on your computer, you download the model yourself, once
        raise RuntimeError(f"{model} is not downloaded yet. Run this command in a terminal (it downloads {size}), "
                           f"then run this cell again:\n\n    ollama pull {model}\n")
    if model not in installed:
        print(f"WARNING: {model} is not here yet. It is downloaded now ({size}), this can take several minutes.")
        request = urllib.request.Request(url + "/api/pull", data=json.dumps({"model": model}).encode())
        shown = -1
        for line in urllib.request.urlopen(request):
            status = json.loads(line)
            if "error" in status:
                raise RuntimeError(status["error"])
            if status.get("total"):
                percent = int(100 * status.get("completed", 0) / status["total"]) // 10 * 10
                if percent != shown:
                    print(f"  {percent} %", end="  ")
                    shown = percent
        print("\nDownload finished.")
    return url + "/v1/"


def wait_until_awake(base_url, api_key):
    """A sleeping Hugging Face endpoint answers 503 until it has started; we wait for it."""
    request = urllib.request.Request(base_url + "/models", headers={"Authorization": f"Bearer {api_key}"})
    for attempt in range(40):  # up to about 6 minutes
        try:
            urllib.request.urlopen(request, timeout=10)
            return
        except urllib.error.HTTPError as error:
            if error.code == 403:
                raise RuntimeError("The endpoint answers 403: it is paused (the course endpoints only run during the seminar), "
                                   "or your key has no access to it. Choose another provider, or check HF_GATEWAY_KEY and the address in .env.")
            if error.code != 503:
                raise RuntimeError(f"The endpoint answers {error.code}. Check the key and the address.")
        except OSError:
            pass
        if attempt == 0:
            print("The endpoint is asleep and starts now. This takes a few minutes ...")
        time.sleep(8)
    raise RuntimeError("The endpoint did not wake up. Try again in a few minutes.")


def setup_model(model):
    """Checks {"model_name": ..., "provider": ...}, sets the address and the key, and prints what to keep in mind.
    Returns base_url, api_key and the model name for ChatOpenAI."""
    name = model.get("model_name", "").strip()
    provider = model.get("provider", "").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}. Choose one of: {', '.join(PROVIDERS)}.")
    if provider in ("ollama", "colab", "huggingface") and name not in QWEN:
        raise ValueError(f"The provider {provider!r} has these models: {', '.join(QWEN)}. Not {name!r}.")
    notes = []

    if provider in ("ollama", "colab"):
        if provider == "ollama" and IN_COLAB:
            raise ValueError("This notebook runs on Google Colab: use the provider 'colab'.")
        if provider == "colab" and not IN_COLAB:
            raise ValueError("The provider 'colab' only works on Google Colab. On your own computer, use 'ollama'.")
        if provider == "colab" and subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
            notes.append("This Colab runtime has no GPU: answers take long. Better: Runtime > Change runtime type > T4 GPU, then run the notebook again.")
        if provider == "ollama":
            notes.append("The model runs on this computer. Without a GPU, it runs on the CPU, and answers take longer.")
        base_url, api_key = start_ollama(name, QWEN[name][0], colab=(provider == "colab")), "ollama"  # Ollama needs no key
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): Ollama returns only one.")

    elif provider == "huggingface":
        _, env_name, name = QWEN[name]  # name: the model on Hugging Face
        base_url = value(env_name, f"Paste the address of the Hugging Face endpoint for {model['model_name']}: ").rstrip("/")
        if "<" in base_url:
            raise ValueError(f"The address for {model['model_name']} is still a placeholder in .env ({env_name}). "
                             "Paste the address from the course there, or choose another model or provider.")
        if not base_url.endswith("/v1"):
            base_url += "/v1"  # the OpenAI-compatible API of the endpoint
        api_key = value("HF_GATEWAY_KEY", "Paste your Hugging Face key (for a public endpoint, just press Enter): ", secret=True)
        api_key = api_key or "none"  # a public endpoint needs no key, but the client needs some text
        wait_until_awake(base_url, api_key)
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): the endpoint refuses them (HTTP 400).")

    elif provider == "ai-gateway":
        if IN_COLAB:
            notes.append("Colab runs outside the university network and usually cannot reach the AI-Gateway.")
        base_url = os.getenv("AI_GATEWAY_BASE") or "https://ai-gateway.uni-paderborn.de/v1"
        api_key = value("AI_GATEWAY_KEY", "Paste your AI-Gateway key: ", secret=True)
        if name not in GATEWAY_QWEN:
            notes.append(f"{name!r} is not one of the Qwen models of the course ({', '.join(GATEWAY_QWEN)}). "
                         "llm.root_client.models.list() lists the models of your account.")
        notes.append("Thinking cannot be switched with reasoning_effort here (the gateway ignores it): the model name decides. "
                     "gwdg.qwen3.8-27b-no-think = off, gwdg.qwen3.8-27b = on.")
        notes.append("Only in the university network or the VPN. About 10 requests per minute: if a cell pauses, the client waits and continues.")

    else:  # another OpenAI-compatible API
        if not name:
            raise ValueError("Give the model name of your provider as model_name.")
        base_url = value("OTHER_BASE", "Base URL of the API (e.g. https://api.openai.com/v1/): ")
        api_key = value("OTHER_KEY", "Your API key: ", secret=True)
        notes.append("Check in the documentation of your provider how thinking is switched; "
                     "the extra_body of ChatOpenAI may need other fields than reasoning_effort.")

    print(f"Provider: {provider} | model: {name}")  # the key is never printed
    print("Keep in mind:")
    for note in notes:
        print("  -", note)
    return base_url, api_key, name

**Before every `ChatOpenAI`, we call `setup_model()`** with the provider and the model, for example `setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})`. It checks whether the combination exists, sets the address and the key, and prints what to keep in mind. It returns three values, which we split directly into `BASE_URL`, `API_KEY`, and `MODEL_NAME`. To switch the model, change the dict there.

In [ ]:
import json
import os
import random
import urllib.request
from glob import glob
from typing import Any, Mapping

import numpy as np
import textworld
import textworld.gym
from langchain_openai import ChatOpenAI
from termcolor import colored
from textworld import EnvInfos
from textworld.gym import Agent
from tqdm.notebook import tqdm

BASE_URL, API_KEY, MODEL_NAME = setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})  # provider and model: see the table in the setup
llm = ChatOpenAI(
    model=MODEL_NAME,  # on the AI-Gateway, the model decides about thinking: gwdg.qwen3.8-27b-no-think (off) or gwdg.qwen3.8-27b (on)
    base_url=BASE_URL,
    api_key=API_KEY,
    temperature=0,
    seed=42,
    extra_body={
        "max_tokens": 1024,  # the token limit (here, because Ollama ignores max_completion_tokens, the name that ChatOpenAI uses)
        "reasoning_effort": "none",  # thinking off; "low", "medium", or "high" switches it on (Ollama, Hugging Face endpoint)
    },
)

if MODEL_NAME.startswith("qwen3.5:") and MODEL_NAME != "qwen3.5:9b":
    print("Note: the basic attempt was made with qwen3.5:9b. Smaller models lose track of the game more often.")

## TextWorld and the First TextWorld Problems Challenge

**TextWorld** is a learning environment for text-based games by Microsoft, built for agents that learn to play them ([Côté et al., 2018](https://doi.org/10.48550/arXiv.1806.11532)). With it, Microsoft published the *First TextWorld Problems Challenge* and its *Cooking Game*: the agent walks through rooms, finds the recipe, finds the ingredients, cuts and cooks them according to the recipe, and eats the meal. It only uses text commands ([Microsoft, 2019](https://www.microsoft.com/en-us/research/blog/first-textworld-problems-the-competition-using-text-based-games-to-advance-capabilities-of-ai-agents/)).

You can try a game in the browser: [TextWorld: try it](https://www.microsoft.com/en-us/research/project/textworld/try-it/).

### The games

We use a small part of the challenge: 20 games for training, 15 for validation, and 17 for testing. They are in the folder `games` next to this notebook. If the folder is missing, for example on Colab, the next cell downloads the games from the course repository.

In [ ]:
# downloads the games, if they are not here yet (for example on Colab)
GAMES_URL = "https://api.github.com/repos/olivermueller/amlta-2026/contents/Session_04/games/"

for split in ["train", "valid", "test"]:
    folder = os.path.join("games", split)
    if glob(os.path.join(folder, "*.z8")):
        continue
    os.makedirs(folder, exist_ok=True)
    files = json.load(urllib.request.urlopen(GAMES_URL + split))  # the list of files in this folder
    for file in tqdm(files, desc=f"downloading games/{split}"):
        if file["name"].endswith((".z8", ".json")):  # the game and its walkthrough; the .ulx files are not needed
            urllib.request.urlretrieve(file["download_url"], os.path.join(folder, file["name"]))

GAMES = {split: sorted(glob(os.path.join("games", split, "*.z8"))) for split in ["train", "valid", "test"]}
for split, files in GAMES.items():
    print(f"{split}: {len(files)} games")

### Playing a game

`play()` runs one game: it asks the agent for a command, sends it to the game, and repeats until the game is over or 50 moves are reached. If the answer of an LLM is too long to be a command, it asks again, at most three times.

In [ ]:
def play(agent, path, max_step=50, seed=42, verbose=True, render=True):
    """Plays one game with the agent. Returns moves, score, normalized score, maximum score, and the messages."""
    random.seed(seed)
    np.random.seed(seed)

    infos_to_request = agent.infos_to_request
    infos_to_request.max_score = True  # needed to normalize the score

    env_id = textworld.gym.register_games([path], request_infos=infos_to_request, max_episode_steps=max_step)
    env = textworld.gym.make(env_id)
    if verbose:
        print(os.path.basename(path))

    obs, infos = env.reset()  # start the game
    if render:
        env.render()

    obs_template_violation = "Your answer did not follow the template. Give the answer as described in your task."

    score, done, moves = 0, False, 0
    while not done:
        command = agent.act(obs, score, done, infos)

        rethink_counter = 0
        while len(command) > 50:  # too long for a command: ask again, at most three times
            command = agent.act(obs_template_violation, score, done, infos)
            rethink_counter += 1
            moves += 1
            if rethink_counter == 3:
                break
        if len(command) > 50:
            command = "I don't know"

        obs, score, done, infos = env.step(command)
        if render:
            env.render()
        moves += 1

    agent.act(obs, score, done, infos)  # let the agent know that the game is over
    env.close()

    norm_score = score / infos["max_score"]
    if verbose:
        print(f"Moves: {moves:3d}\tScore: {score:4d} / {infos['max_score']:4d}\tNormalized score: {norm_score:3.2f}")
    return moves, score, norm_score, infos["max_score"], agent.messages, agent.messages_raw

### You play first

The agent below is **you**: it asks you for every command. The commands look like `read cookbook`, `take knife`, or `open fridge`. Play one game to see what the model has to do.

* You start in the kitchen.
* You read the recipe in the cookbook.
* You find the ingredients (maybe in the fridge), take them and the knife.
* You cut and cook the ingredients according to the recipe, prepare the meal, and eat it.

In [ ]:
class HumanAgent(Agent):
    """An agent that asks you for every command."""

    def __init__(self):
        self.messages = []
        self.messages_raw = []

    @property
    def infos_to_request(self) -> EnvInfos:
        return EnvInfos()

    def act(self, obs: str, score: int, done: bool, infos: Mapping[str, Any]) -> str:
        return input()

In [ ]:
play(HumanAgent(), "games/train/tw-cooking-recipe1+cut-OYmoIYkTvbNcxja.z8", render=True)

## The LLM Agent

**In short: every move is one request, with a system prompt, the last moves, and the current state of the game.**

`LLMAgent` builds the messages for every move:

| Part | What it contains |
|---|---|
| **system prompt** | the rules of the game, the same for every move |
| **examples** | optional few-shot examples (4.03), as earlier `user` and `assistant` messages |
| **history** | the last `history_length` moves: what the agent saw and what it answered |
| **current state** | your `prompt`, filled with the game's feedback, the recipe, the inventory, and the room |

The agent takes the text behind the last `:` of the answer as the command, so answers like `NEXT COMMAND: take knife` work, too.

In [ ]:
RECIPE_START = 'You open the copy of "Cooking: A Modern Approach (3rd Ed.)" and start reading:'


class LLMAgent(Agent):
    """An agent that asks the LLM for every command."""

    def __init__(self, llm, prompt, history_length, system_message, examples, env_infos={}, render=False):
        self.llm = llm
        self.prompt = prompt
        self.history_length = history_length
        self.examples = list(examples)
        self.messages = [{"role": "system", "content": system_message}] + self.examples
        self.env_infos = EnvInfos(**env_infos)
        self.render = render
        self.recipe = "You have not read the recipe yet."
        self.messages_raw = []

    @property
    def infos_to_request(self) -> EnvInfos:
        return self.env_infos

    def act(self, obs: str, score: int, done: bool, infos: Mapping[str, Any]) -> str:
        inventory = (infos.get("inventory") or "").strip()
        description = (infos.get("description") or "").strip()
        feedback = obs.strip()
        if RECIPE_START in feedback:  # the agent remembers the recipe once it has read it
            self.recipe = feedback.replace(RECIPE_START, "").strip()

        prompt = self.prompt.format(feedback=feedback, recipe=self.recipe, inventory=inventory, description=description)
        self.messages_raw.append({"role": "user", "feedback": feedback, "recipe": self.recipe, "inventory": inventory, "description": description})
        self.messages.append({"role": "user", "content": prompt})
        if self.render:
            print(prompt)

        head = self.messages[: 1 + len(self.examples)]  # the system prompt and the examples
        history = self.messages[1 + len(self.examples):][-(2 * self.history_length + 1):]  # the last moves and the current state
        answer = self.llm.invoke(head + history).content

        self.messages.append({"role": "assistant", "content": answer})
        self.messages_raw.append({"role": "assistant", "content": answer})
        if self.render:
            print(colored("> " + answer, "yellow"))
        return answer.split(":")[-1].strip()

### Evaluating a prompt

`evaluate_agent()` plays a list of games with the same prompts and counts:

* **wins:** games where the agent reached the maximum score,
* **average normalized score:** the share of all points that the agent collected.

It saves every game, with all messages, in a JSON file in the folder `results`, so you can read later where the agent got stuck.

In [ ]:
HISTORY_LENGTH = 3  # the number of earlier moves in every request
ENV_INFOS = {"description": True, "inventory": True}  # what the game tells the agent in every move


def evaluate_agent(prompt, system_message, examples, games, results_path, history_length=HISTORY_LENGTH, model=None, render_agent=False, render=False):
    """Plays all games with the same prompts, saves the results, and prints the wins and the average normalized score.
    model: another chat model, for example with other settings; without it, the agent uses llm."""
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    results, score_sum, max_score_sum, wins = {}, 0, 0, 0
    for path in tqdm(games):
        config = json.load(open(path.replace(".z8", ".json")))
        walkthrough = config.get("extras", {}).get("walkthrough", [])
        agent = LLMAgent(model or llm, prompt, history_length, system_message, examples, ENV_INFOS, render=render_agent)
        moves, score, norm_score, max_score, messages, messages_raw = play(agent, path, render=render)
        results[os.path.basename(path)] = {
            "moves": moves, "min_moves": len(walkthrough), "score": score, "norm_score": norm_score, "max_score": max_score,
            "recipe": config.get("extras", {}).get("recipe", ""), "walkthrough": walkthrough,
            "messages": messages, "messages_raw": messages_raw,
        }
        with open(results_path, "w") as file:
            json.dump(results, file)
        score_sum += score
        max_score_sum += max_score
        wins += score == max_score
    print(f"Wins: {wins} / {len(games)}")
    print(f"Average normalized score: {score_sum / max_score_sum:.2f}")

## The Basic Attempt

**In short: a system prompt with the rules of the game, and a short prompt with the current state.**

The system prompt is the same for every move. It describes the game like a manual: the entities, the commands, and the process. The prompt for every move only contains what changes: the feedback of the game, the recipe, the inventory, and the room.

We start with **one** game and watch every move. The yellow lines are the answers of the model.

In [ ]:
SYSTEM_MESSAGE = """You are the player of a text-based cooking game.

These are the important game entities: 
cookbook:       Cookbook that contains the recipe.
<ingredient>:   Ingredient you can take, drop, cut, or cook according to the recipe.
<container>:    Container you can open, which can contain ingredients.
<door>:         Door you can open.
knife:          Knife to slice, dice, chop ingredients.
stove:          Stove to fry ingredients.
oven:           Oven to roast ingredients.
bbq:            BBQ to grill ingredients.
meal:           Meal you can eat.

These are the commands to interact with the game:
look
open <container>
close <container>
open <door>
close <door>
inventory
take <ingredient>
take knife
read cookbook
slice <ingredient> with knife
dice <ingredient> with knife
chop <ingredient> with knife
cook <ingredient> with stove    (fry an ingredient)
cook <ingredient> with oven     (roast an ingredient)
cook <ingredient> with bbq      (grill an ingredient)
prepare meal
eat meal

This is the basic game process:
Navigate to the kitchen and read the cookbook.
Take all ingredients according to the recipe.
Follow the recipe.
Eat the meal.

Your main task:
You provide answers to game engine queries following the command templates and game process above.
Never provide answers deviating from the command templates above - unless explicitly asked by the game engine.
Provide only one command at a time - unless the game engine explicitly asks to do something else.
Do not make tool calls!!! Use the pre-defined commands as text."""

PROMPT = """# Current game state:
## {feedback}

## {recipe}

## {inventory}

## You are in the {description}"""

EXAMPLES = []  # no few-shot examples in the basic attempt

evaluate_agent(PROMPT, SYSTEM_MESSAGE, EXAMPLES, GAMES["train"][0:1], "results/train/BASIC_one_game.json", render_agent=True)

Let's look at the outcome: which commands worked, and where did the model get stuck?

Now all 20 training games, without printing every move. This takes a while.

In [ ]:
evaluate_agent(PROMPT, SYSTEM_MESSAGE, EXAMPLES, GAMES["train"], "results/train/BASIC.json")

### Validation

You develop your prompts on the **training** games. The **validation** games show whether an improvement also holds on games the prompt was not tuned for. The **test** games come only at the very end, once.

In [ ]:
evaluate_agent(PROMPT, SYSTEM_MESSAGE, EXAMPLES, GAMES["valid"], "results/valid/BASIC.json")

## Your Turn

**In short: start from a copy of the basic attempt and improve it with what you learned in 4.01 to 4.04.**

The cell below is a copy of the basic attempt. Change whatever you want, try it on one game, then on all training games, and compare it with the basic attempt. Check your best version on the validation games.

### How to work: best practices for prompt engineering

1. **Start general, then tailor.** First a meta-prompt skeleton (role, task, rules, process, output format). Add task-specific rules only for failures you have actually seen.
2. **System prompt for what stays the same:** role, behavior, constraints, allowed actions, output format. **User prompt for what changes:** the current state and the input.
3. **Use structure.** Headings, lists, and delimiters; mark input as data.
4. **Fix the output format exactly** and parse it robustly, with a plan for violations (ask again, or a fallback).
5. **Give the information the model needs, not all of it.** Keep the context short and relevant.
6. **Let the model reason when the task needs planning,** but limit the length and measure the cost.
7. **Examples show format and style.** Take them from training data only, vary them, and watch for bias.
8. **Clear, positive instructions** instead of shouting ("!!!", capitals) and vague bans.
9. **Read failures before changing anything,** change one thing at a time, and keep a log of versions.
10. **Train, validation, test:** develop on train, choose on validation, report on test once.
11. **Test again after switching the model or provider.** Prompts don't carry over automatically.
12. **Let the model improve the prompt (meta prompting),** but decide by the score.

### What you learned (4.01 to 4.04)

| Technique | From | An idea for this game |
|---|---|---|
| Chain-of-thought | 4.01 | let the model reason before the command, e.g. `REASONING: ... NEXT COMMAND: ...` (the agent takes the text behind the last `:`), and raise `max_tokens` in `MY_LLM` |
| Few-shot examples | 4.03 | earlier `user` and `assistant` messages in `MY_EXAMPLES`, e.g. a few moves from the walkthrough of a training game (`walkthrough` in the game's `.json` file) |
| Self-consistency | 4.04, Part 1 | several answers for every move and a majority vote; needs a change in `LLMAgent.act()` and costs several times the tokens |
| Meta prompt | 4.04, Part 2 | describe the structure of a good move instead of showing examples |
| System prompt | 4.04, Part 3 | role, rules, and output format: everything that stays the same in every move |
| Structure | 4.04, Part 4 | `###` blocks, and delimiters around the game's feedback: it is data, not instructions |
| Automatic prompt engineering | 4.04, Part 5 | give the model your system prompt and a lost game from `results/train/...json`, and let it write a better system prompt with `llm.invoke()` |

### What you can configure

| Setting | What it changes |
|---|---|
| `MY_SYSTEM_MESSAGE` | the rules for every move |
| `MY_PROMPT` | the state of the game in every move; the placeholders `{feedback}`, `{recipe}`, `{inventory}`, and `{description}` are filled by the agent |
| `MY_EXAMPLES` | few-shot examples, as a list of `{"role": ..., "content": ...}` messages |
| `MY_HISTORY_LENGTH` | the number of earlier moves in every request (basic attempt: 3) |
| `MY_LLM` | the model and its settings: where it runs (`setup_model(...)` right before it), `temperature`, the token limit (`max_tokens`), and thinking (`reasoning_effort`); on the AI-Gateway, thinking needs the model `gwdg.qwen3.8-27b` |
| the games | `GAMES["train"][0:3]` for a quick test; `render_agent=True` prints every move |
| more information from the game | TextWorld can tell the agent more, for example the allowed commands (`"admissible_commands": True` in `ENV_INFOS`, see [EnvInfos](https://textworld.readthedocs.io/en/stable/textworld.html#textworld.core.EnvInfos)); then add it to the prompt in `LLMAgent.act()` |

Change one thing at a time, and note the wins of every version. Otherwise you will not know which change helped. Run your best version twice: even with `temperature=0`, two runs can differ a little.

In [ ]:
# a copy of the basic attempt: change whatever you want
MY_SYSTEM_MESSAGE = """You are the player of a text-based cooking game.

These are the important game entities: 
cookbook:       Cookbook that contains the recipe.
<ingredient>:   Ingredient you can take, drop, cut, or cook according to the recipe.
<container>:    Container you can open, which can contain ingredients.
<door>:         Door you can open.
knife:          Knife to slice, dice, chop ingredients.
stove:          Stove to fry ingredients.
oven:           Oven to roast ingredients.
bbq:            BBQ to grill ingredients.
meal:           Meal you can eat.

These are the commands to interact with the game:
look
open <container>
close <container>
open <door>
close <door>
inventory
take <ingredient>
take knife
read cookbook
slice <ingredient> with knife
dice <ingredient> with knife
chop <ingredient> with knife
cook <ingredient> with stove    (fry an ingredient)
cook <ingredient> with oven     (roast an ingredient)
cook <ingredient> with bbq      (grill an ingredient)
prepare meal
eat meal

This is the basic game process:
Navigate to the kitchen and read the cookbook.
Take all ingredients according to the recipe.
Follow the recipe.
Eat the meal.

Your main task:
You provide answers to game engine queries following the command templates and game process above.
Never provide answers deviating from the command templates above - unless explicitly asked by the game engine.
Provide only one command at a time - unless the game engine explicitly asks to do something else.
Do not make tool calls!!! Use the pre-defined commands as text."""

MY_PROMPT = """# Current game state:
## {feedback}

## {recipe}

## {inventory}

## You are in the {description}"""

MY_EXAMPLES = []

MY_HISTORY_LENGTH = 3

BASE_URL, API_KEY, MODEL_NAME = setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})  # provider and model: see the table in the setup
MY_LLM = ChatOpenAI(
    model=MODEL_NAME,  # on the AI-Gateway, the model decides about thinking: gwdg.qwen3.8-27b-no-think (off) or gwdg.qwen3.8-27b (on)
    base_url=BASE_URL,
    api_key=API_KEY,
    temperature=0,
    seed=42,
    extra_body={
        "max_tokens": 1024,  # the token limit (here, because Ollama ignores max_completion_tokens, the name that ChatOpenAI uses)
        "reasoning_effort": "none",  # thinking off; "low", "medium", or "high" switches it on (Ollama, Hugging Face endpoint)
    },
)

# first a quick test on one game, with every move printed
evaluate_agent(MY_PROMPT, MY_SYSTEM_MESSAGE, MY_EXAMPLES, GAMES["train"][0:1], "results/train/MY_STRATEGY_one_game.json",
               history_length=MY_HISTORY_LENGTH, model=MY_LLM, render_agent=True)

In [ ]:
# all training games: compare with the basic attempt
evaluate_agent(MY_PROMPT, MY_SYSTEM_MESSAGE, MY_EXAMPLES, GAMES["train"], "results/train/MY_STRATEGY.json",
               history_length=MY_HISTORY_LENGTH, model=MY_LLM)

In [ ]:
# your best version on the validation games
evaluate_agent(MY_PROMPT, MY_SYSTEM_MESSAGE, MY_EXAMPLES, GAMES["valid"], "results/valid/MY_STRATEGY.json",
               history_length=MY_HISTORY_LENGTH, model=MY_LLM)

### Test, once at the end

Only when you have chosen your best version: evaluate the basic attempt and your version on the test games.

In [ ]:
evaluate_agent(PROMPT, SYSTEM_MESSAGE, EXAMPLES, GAMES["test"], "results/test/BASIC.json")
evaluate_agent(MY_PROMPT, MY_SYSTEM_MESSAGE, MY_EXAMPLES, GAMES["test"], "results/test/MY_STRATEGY.json",
               history_length=MY_HISTORY_LENGTH, model=MY_LLM)

## What You Learned

* An LLM can act as an **agent** in a task with many steps: every move is one request with the system prompt, the last moves, and the current state.
* The **system prompt** holds what stays the same (rules, commands, process); the **prompt** of every move holds what changes.
* Develop on the **training** games, choose on the **validation** games, and report on the **test** games, once.
* Read the saved transcripts in `results` before you change a prompt: they show where the agent got stuck.
* Sometimes even good prompts are not enough. Then fine-tuning can help (Session 8).